# Positional encoding: ручной ID/OOD-анализ

Ноутбук не запускает обучение и не делает sweep. В ячейке **Runs** вручную указываются готовые checkpoint'ы, задача и distributions. Для каждого запуска сохраняются:

- accuracy и loss на каждом split;
- число параметров и PE-параметров из checkpoint metadata;
- per-example accuracy по длине, абсолютной и нормированной позиции, relative distance;
- CSV-таблицы и обычные matplotlib-графики.

Зависимости: `torch`, `numpy`, `pandas`, `matplotlib` и код текущего репозитория.

In [ ]:
from __future__ import annotations

import os
import sys
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def find_repo(start: Path) -> Path:
    candidates = [Path(os.environ.get("REPO_DIR", "")), start, *start.parents]
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if (candidate / "models").is_dir() and (candidate / "generator").is_dir():
            return candidate
    raise FileNotFoundError("Не найден корень репозитория; задайте REPO_DIR")


REPO_DIR = find_repo(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from generator import get_generator
from models import get_model


def load_checkpoint(out_dir, device="cpu"):
    checkpoint = torch.load(Path(out_dir) / "ckpt.pt", map_location=device)
    model_name = checkpoint.get("config", {}).get("model", "positional")
    ModelConfig, Model = get_model(model_name)
    model = Model(ModelConfig(**checkpoint["model_args"]))
    state_dict = checkpoint["model"]
    for key in list(state_dict):
        if key.startswith("_orig_mod."):
            state_dict[key[len("_orig_mod."):]] = state_dict.pop(key)
    model.load_state_dict(state_dict)
    return model.eval().to(device), checkpoint


def build_generator(checkpoint, dataset_override=None, gen_param_overrides=None, seed=1337):
    dataset = dataset_override or checkpoint.get("config", {}).get("dataset")
    params = dict(checkpoint.get("config", {}).get("gen_params", {}))
    params.update(gen_param_overrides or {})
    params["seed"] = seed
    return get_generator(dataset, params)


@torch.no_grad()
def evaluate_accuracy(model, generator, n_eval, block_size, device, ctx):
    items = generator.sample(n_eval)
    x_np, y_np = generator.collate(items, block_size)
    x = torch.from_numpy(x_np).to(device)
    y = torch.from_numpy(y_np).to(device)
    with ctx:
        logits, loss = model(x, y)
    mask = y != -1
    predictions = logits.argmax(dim=-1)
    row_correct = ((predictions == y) | ~mask).all(dim=1)
    return {
        "accuracy": float(row_correct.float().mean()),
        "loss": float(loss),
        "n_eval": len(items),
        "row_correct": row_correct.cpu().numpy(),
        "items": items,
    }

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
RESULTS_DIR = Path(os.environ.get("PE_RESULTS_DIR", REPO_DIR / "analysis_results" / "positional"))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_DIR)
print("device:", DEVICE)
print("results:", RESULTS_DIR)


## Runs

Каждая запись — один обученный checkpoint и набор контролируемых split'ов. `checkpoint` указывает на каталог с `ckpt.pt`. Параметры каждого split передаются непосредственно в `PositionalLabGenerator`.

Оставь `enabled=False` у примера и добавь свои записи. Для честной ablation используй одинаковые split'ы, `n_eval` и seeds для сравниваемых PE.

In [ ]:
RUNS = [
    {
        "enabled": False,
        "name": "rope-relative-copy",
        "checkpoint": REPO_DIR / "out" / "pe_experiments" / "relative_offset_copy" / "rope" / "seed_1337",
        "task": "relative_offset_copy",
        "n_eval": 1000,
        "seed": 2026,
        "splits": {
            "ID": {"length_range": (16, 32), "distance_range": (4, 8)},
            "OOD-1": {"length_range": (33, 64), "distance_range": (9, 16)},
            "OOD-2": {"length_range": (65, 128), "distance_range": (17, 32)},
        },
    },
]

ACTIVE_RUNS = [run for run in RUNS if run.get("enabled", True)]
print(f"active runs: {len(ACTIVE_RUNS)}")
for run in ACTIVE_RUNS:
    print(run["name"], "->", run["checkpoint"])


## Evaluation

Один и тот же checkpoint загружается один раз. Для каждого split создаётся независимый, но воспроизводимый поток примеров. `evaluate_accuracy` использует тот же exact-match критерий и `collate`, что и training loop.

In [ ]:
def amp_context():
    if DEVICE != "cuda":
        return nullcontext()
    return torch.autocast(device_type="cuda", dtype=DTYPE)


def positional_name(checkpoint: dict) -> str:
    metadata = checkpoint.get("run_metadata", {})
    if metadata.get("positional_encoding"):
        return metadata["positional_encoding"]
    config = checkpoint.get("config", {})
    model = config.get("model", "base")
    if model == "positional":
        return config.get("pos_encoding", "nope")
    return "wpe" if model == "base" else model


def evaluate_run(run: dict):
    model, checkpoint = load_checkpoint(str(Path(run["checkpoint"])), device=DEVICE)
    pe = positional_name(checkpoint)
    summaries, details = [], []

    for split_index, (split, overrides) in enumerate(run["splits"].items()):
        params = dict(overrides)
        generator = build_generator(
            checkpoint,
            dataset_override=run["task"],
            gen_param_overrides=params,
            seed=int(run.get("seed", 2026)) + split_index,
        )
        result = evaluate_accuracy(
            model=model,
            generator=generator,
            n_eval=int(run.get("n_eval", 1000)),
            block_size=model.config.block_size,
            device=DEVICE,
            ctx=amp_context(),
        )
        metadata = checkpoint.get("run_metadata", {})
        summaries.append({
            "run": run["name"],
            "positional_encoding": pe,
            "task": run["task"],
            "split": split,
            "accuracy": result["accuracy"],
            "loss": result["loss"],
            "n_eval": result["n_eval"],
            "number_of_parameters": metadata.get("number_of_parameters", sum(p.numel() for p in model.parameters())),
            "positional_parameters": metadata.get("positional_parameters", {}).get("total", np.nan),
            "distribution": repr(overrides),
        })
        for item, correct in zip(result["items"], result["row_correct"]):
            details.append({
                "run": run["name"],
                "positional_encoding": pe,
                "task": run["task"],
                "split": split,
                "correct": bool(correct),
                **item.metadata,
            })
        print(f'{run["name"]:28s} {split:8s} accuracy={result["accuracy"]:.4f} loss={result["loss"]:.4f}')

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return summaries, details


In [ ]:
all_summaries, all_details = [], []
for run in ACTIVE_RUNS:
    summaries, details = evaluate_run(run)
    all_summaries.extend(summaries)
    all_details.extend(details)

summary = pd.DataFrame(all_summaries)
detail = pd.DataFrame(all_details)
display(summary)


## Общий ID/OOD-график

In [ ]:
if not summary.empty:
    pivot = summary.pivot_table(index="split", columns="run", values="accuracy")
    ax = pivot.plot(marker="o", figsize=(10, 4.8))
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("Exact-match accuracy")
    ax.set_title("Positional OOD generalization")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("Добавьте хотя бы один enabled run.")


## Position-sensitive разрезы

Для дискретных осей строится средняя exact-match accuracy. Нормированная позиция дополнительно разбивается на десять одинаковых интервалов `[0, 1]`. Пустые metadata-поля конкретной задачи автоматически пропускаются.

In [ ]:
def plot_axis(frame: pd.DataFrame, axis: str, *, bins=None):
    if frame.empty or axis not in frame or frame[axis].notna().sum() == 0:
        print(f"skip {axis}: no data")
        return
    work = frame.dropna(subset=[axis]).copy()
    x = axis
    if bins is not None:
        x = axis + "_bin"
        work[x] = pd.cut(work[axis], bins=bins, include_lowest=True)
    grouped = work.groupby([x, "run"], observed=True)["correct"].mean().unstack("run")
    ax = grouped.plot(marker="o", figsize=(10, 4.5))
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("Exact-match accuracy")
    ax.set_title(f"Accuracy vs {axis}")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_axis(detail, "sequence_length")
plot_axis(detail, "absolute_target_position")
plot_axis(detail, "normalized_target_position", bins=np.linspace(0, 1, 11))
plot_axis(detail, "relative_distance")


## Экспорт

Две таблицы сохраняются отдельно: агрегат по run/split и per-example данные для собственных графиков или статистических тестов.

In [ ]:
SUMMARY_CSV = RESULTS_DIR / "positional_summary.csv"
DETAIL_CSV = RESULTS_DIR / "positional_per_example.csv"

summary.to_csv(SUMMARY_CSV, index=False)
detail.to_csv(DETAIL_CSV, index=False)
print(SUMMARY_CSV)
print(DETAIL_CSV)


## Подсказка по лабораторным задачам

| PE | Favorable task | Breaking task |
|---|---|---|
| NoPE | `content_addressed_retrieval` | `absolute_position_parity` |
| WPE | `absolute_position_parity` на seen positions | `absolute_position_extrapolation` |
| RoPE | `relative_offset_copy` | `unseen_long_relative_offset` |
| ALiBi | `local_latest_update` | `distant_retrieval` |
| Relative bias | `bucketed_relative_distance` | `exact_distance_beyond_bucket` |
| CoPE | `selective_count` | `dense_absolute_position` |
| CAPE | `context_adaptive_retrieval` | `context_correlation_flip` |
| FoPE | `periodic_distance` | `nonperiodic_absolute_threshold` |

Это гипотезы, а не заранее гарантированные результаты. Для корректного сравнения меняй только PE и держи backbone, optimizer, число шагов, данные и seed одинаковыми.